# 1 - Config & Import

## 1.1 - Config

In [1]:
# 🔧 config import
import os
from core.config.logger_config import colored_logger
logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

[2025-10-26 11:20:55] INFO - 726159153.py - Logger initialized (Notebook)


## 1.2 - Import

In [2]:
# 📦 External libs
import importlib
import torch
import torch.nn as nn
from torchinfo import summary
import torch.optim as optim

# 📂 Internal modules
from Class import Data, Models, Trainer
import Class.Data.Features
import Class.Data.Labels
import Class.Data.Preprocessing
import Class.Data.DataAnalysis


def reload_modules():
    importlib.reload(Class.Data.DataFetcher)
    importlib.reload(Class.Data.Features)
    importlib.reload(Class.Data.Labels)
    importlib.reload(Class.Data.DataAnalysis)
    importlib.reload(Class.Data.Preprocessing)
    importlib.reload(Class.Models.LSTMModel)
    importlib.reload(Class.Trainer.LSTMTrainer)

ModuleNotFoundError: No module named 'Class'

# 2.5  - Preprocessing

### Variables

In [240]:
# Step 0  Initialize Processing data
print(Labels_Data.data.columns)
train_test_data = pd.DataFrame({"Feature_1": Labels_Data.data['VWAP_5m'],
                                "Feature_2": Labels_Data.data["London_Open"],
                                "Feature_3": Labels_Data.data["NY_Open"],
                                "Feature_4": Labels_Data.data["HK_Open"],
                                "Feature_5": Labels_Data.data['Dif_Low_Pivot'],
                                "Feature_6": Labels_Data.data['Dif_High_Pivot'],
                                "Feature_7": Labels_Data.data['Dif_Volume_Low_Pivot'],
                                "Feature_8": Labels_Data.data['Dif_Volume_High_Pivot'],

                                "Label_1": Labels_Data.data['Low_Below_Pivot'],
                                "Label_2": Labels_Data.data['High_Above_Pivot'],
                                "Label_3": Labels_Data.data['VWAP_Below_Volume_Low'],
                                "Label_4": Labels_Data.data['VWAP_Above_Volume_High'],
                                }).dropna()

# Step 1: Create Train/Test data
lookback = 50
size_test_prct = 0.3

# Step 2: Suggest BatchSize
feature_dim = 9
label_dim = 4
lookback = 50
reserved_ram_gb = 1.0
hidden_dim = [64, 64]

# Step 3: Create DataLoaders
val_ratio = 0.2

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize',
       'VWAP_5m', 'London_Open', 'NY_Open', 'HK_Open', 'Dif_Low_Pivot',
       'Dif_High_Pivot', 'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min',
       'Volume_Low_Pivot', 'Volume_High_Pivot', 'Dif_Volume_Low_Pivot',
       'Dif_Volume_High_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')


### Process

In [241]:
# Step 0  Initialize Processing data
Preprocessing_Data = Class.Data.Preprocessing.Preprocessing(train_test_data)

# Step 1: Create Train/Test data
Preprocessing_Data.create_train_test_data(lookback,size_test_prct)

# Step 2: Suggest BatchSize
Preprocessing_Data.suggest_batch_size(feature_dim, label_dim, lookback, reserved_ram_gb, hidden_dim)

# Step 3: Create DataLoaders
Preprocessing_Data.create_dataloaders(val_ratio)

[2025-06-15 17:25:01] INFO - Preprocessing.py - ✅ Using device: cpu
[2025-06-15 17:25:01] INFO - Preprocessing.py - Starting creation of train/test data (PyTorch).
[2025-06-15 17:25:01] INFO - Preprocessing.py - Selected 8 feature(s) and 4 label(s).
[2025-06-15 17:25:01] INFO - Preprocessing.py - Features normalized (min-max).
[2025-06-15 17:25:01] INFO - Preprocessing.py - Generated 50 sequences of length 50.
[2025-06-15 17:25:01] INFO - Preprocessing.py - Train/Test split: 35 train / 15 test samples.
[2025-06-15 17:25:01] INFO - Preprocessing.py - ✅ Train/test tensors successfully created (PyTorch).
[2025-06-15 17:25:01] INFO - Preprocessing.py - Starting batch size estimation (PyTorch)...
[2025-06-15 17:25:01] INFO - Preprocessing.py - Creating PyTorch DataLoaders...
[2025-06-15 17:25:01] INFO - Preprocessing.py - Train/Val split: 28 train / 7 val samples
[2025-06-15 17:25:01] INFO - Preprocessing.py - ✅ PyTorch DataLoaders created successfully.


In [242]:
pd.set_option('display.max_rows', None)     # Affiche toutes les lignes
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes
pd.set_option('display.width', 1000)        # Largeur max pour éviter le retour à la ligne
print(Preprocessing_Data.x_train)  # ou simplement train_test_data
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')

tensor([[[1.0000, 1.0000, 0.0000,  ..., 0.4230, 0.2514, 0.0109],
         [0.7529, 1.0000, 0.0000,  ..., 0.5992, 0.0000, 0.3262],
         [0.6012, 1.0000, 0.0000,  ..., 0.6984, 0.2742, 0.5197],
         ...,
         [0.3119, 0.0000, 1.0000,  ..., 0.8333, 0.4029, 0.2731],
         [0.3235, 0.0000, 1.0000,  ..., 0.8405, 0.4147, 0.0738],
         [0.3937, 0.0000, 1.0000,  ..., 0.8183, 0.4861, 0.1688]],

        [[0.7529, 1.0000, 0.0000,  ..., 0.5992, 0.0000, 0.3262],
         [0.6012, 1.0000, 0.0000,  ..., 0.6984, 0.2742, 0.5197],
         [0.6042, 1.0000, 0.0000,  ..., 0.6976, 0.2773, 0.5159],
         ...,
         [0.3235, 0.0000, 1.0000,  ..., 0.8405, 0.4147, 0.0738],
         [0.3937, 0.0000, 1.0000,  ..., 0.8183, 0.4861, 0.1688],
         [0.3721, 0.0000, 1.0000,  ..., 0.8246, 0.4642, 0.1963]],

        [[0.6012, 1.0000, 0.0000,  ..., 0.6984, 0.2742, 0.5197],
         [0.6042, 1.0000, 0.0000,  ..., 0.6976, 0.2773, 0.5159],
         [0.7059, 1.0000, 0.0000,  ..., 0.6270, 0.3807, 0.

### Verification

In [243]:
print("x_train shape:", Preprocessing_Data.x_train.shape)
print("y_train shape:", Preprocessing_Data.y_train.shape)
print("x_test shape:", Preprocessing_Data.x_test.shape)
print("y_test shape:", Preprocessing_Data.y_test.shape)

for row in Preprocessing_Data.report_lines:
    print(row)

for x_batch, y_batch in Preprocessing_Data.train_loader.take(1):
    print("Train batch shape:", x_batch.shape, y_batch.shape)

for x_batch, y_batch in Preprocessing_Data.val_loader.take(1):
    print("Val batch shape:", x_batch.shape, y_batch.shape)

x_train shape: torch.Size([35, 50, 8])
y_train shape: torch.Size([35, 4])
x_test shape: torch.Size([15, 50, 8])
y_test shape: torch.Size([15, 4])
--- Batch Size Estimation Report (PyTorch) ---
Total RAM (GB):         7.78
Reserved RAM (GB):      1.0
Usable RAM (GB):        6.78
Samples (n_samples):    35
Features per timestep:  9
Labels (n_labels):      4
Lookback (timesteps):   50
Hidden dimensions:      [64, 64] (layers: 2)
Bytes/sample:           51.77 KB
Max batch size (RAM):   137257.0
Max batch size (CPU):   64
Using GPU:              No
Final suggested batch:  32
--------------------------------------------------


AttributeError: 'DataLoader' object has no attribute 'take'

# 3 - Model

## 3.1 - Varibles

In [244]:
reload_modules()

# Set dimensions based on preprocessing
input_dim = Preprocessing_Data.x_train.shape[2]
output_dim = Preprocessing_Data.y_train.shape[1]
hidden_dim = 128  # Increased
num_layers = 2

dropout = 0.3  # Increased dropout

[2025-06-15 17:25:35] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:25:35] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:25:35] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:25:35] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:25:35] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:25:35] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:25:35] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)


## 3.2 - Model Initialization

In [245]:
# Initialize model
model = Class.Models.LSTMModel.LSTMModel(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    num_layers=num_layers,
    dropout=dropout
).to(Preprocessing_Data.device)

# Model summary
print(summary(
    model,
    input_size=(Preprocessing_Data.batch_size, Preprocessing_Data.x_train.shape[1], input_dim),
    device=Preprocessing_Data.device
))

# Optional: pos_weight for imbalanced binary labels
total_positives = Preprocessing_Data.y_train.sum(dim=0)
total_negatives = Preprocessing_Data.y_train.shape[0] - total_positives
pos_weight = total_negatives / (total_positives + 1e-6)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(Preprocessing_Data.device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

Layer (type:depth-idx)                   Output Shape              Param #
LSTMModel                                [32, 4]                   --
├─LSTM: 1-1                              [32, 50, 128]             202,752
├─BatchNorm1d: 1-2                       [32, 128]                 256
├─Sequential: 1-3                        [32, 4]                   --
│    └─Linear: 2-1                       [32, 64]                  8,256
│    └─LeakyReLU: 2-2                    [32, 64]                  --
│    └─Dropout: 2-3                      [32, 64]                  --
│    └─Linear: 2-4                       [32, 4]                   260
Total params: 211,524
Trainable params: 211,524
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 324.68
Input size (MB): 0.05
Forward/backward pass size (MB): 1.69
Params size (MB): 0.85
Estimated Total Size (MB): 2.59


# 4 - Training & Visualization

## 4.1 - Variables

In [246]:
reload_modules()

[2025-06-15 17:25:41] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-15 17:25:41] INFO - Features.py - Logger initialized (Features.py)
[2025-06-15 17:25:41] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-15 17:25:41] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-15 17:25:41] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-15 17:25:41] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-15 17:25:41] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)


## 4.2 - Model Trainer

In [247]:
# Instantiate the trainer
trainer = Class.Trainer.LSTMTrainer.LSTMTrainer(
    model=model,
    preprocessing=Preprocessing_Data,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    patience=5  # You can increase to 10 if needed
)

# Train the model
trainer.train(num_epochs=5)


Epoch 01 | Train Loss: 0.6457 | Val Loss: 0.6351 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.6875 | ExactMatch: 0.2857 | HammingLoss: 0.2500
Epoch 02 | Train Loss: 0.6119 | Val Loss: 0.6335 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.6875 | ExactMatch: 0.2857 | HammingLoss: 0.2500
Epoch 03 | Train Loss: 0.5885 | Val Loss: 0.6319 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.6875 | ExactMatch: 0.2857 | HammingLoss: 0.2500
Epoch 04 | Train Loss: 0.5812 | Val Loss: 0.6304 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.6875 | ExactMatch: 0.2857 | HammingLoss: 0.2500
Epoch 05 | Train Loss: 0.5242 | Val Loss: 0.6288 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000 | AUC: 0.7083 | ExactMatch: 0.2857 | HammingLoss: 0.2500


In [ ]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
